# Heat Network Supertargeting Demo

This notebook is the standalone version of the original workbook-driven
heat-network calculation from the HDA workflow.

It shows how to:

1. read thermal-stream data from the Excel sheet
2. convert the rows into `ThermalStream` objects
3. run pinch and supertargeting calculations at a chosen `Delta Tmin`
4. inspect utility targets, pinch temperatures, and area estimates
5. plot composite-curve style summaries from the computed results


## Files Used

- `data/Input Sheet of 20.xlsx`: the thermal-stream workbook
- `heat_network_demo/`: the local helper package used by this notebook
- `images/`: pre-rendered figures from the original workflow

The notebook now imports only local modules from this repository.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

NOTEBOOK_DIR = Path.cwd().resolve()
DEMO_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == 'heat_network_supertargeting' else NOTEBOOK_DIR.parent
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

from heat_network_demo import (
    curve_plot_records,
    parse_xlsx_rows,
    read_streams_from_xlsx,
    replay_notebook_area_algorithm,
    run_supertargeting,
    stream_rows_from_workbook,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
plt.style.use('default')


In [ ]:
INPUT_XLSX = DEMO_DIR / 'data' / 'Input Sheet of 20.xlsx'
IMAGE_DIR = DEMO_DIR / 'images'
DELTA_TMIN_C = 20.0

PROCESS_ONLY_UTILITY_NAMES = {
    'Hot Oil',
    'HP Steam',
    'Cooling Water',
    'Refrigerant 2',
    'Fired Heat (1000)',
}

INPUT_XLSX


## 1. Read the Workbook

The helper uses a lightweight XLSX parser so the notebook does not depend on
a private CES workspace path. The table below is the cleaned thermal-stream
sheet used for the calculations.


In [ ]:
workbook_rows = parse_xlsx_rows(INPUT_XLSX)
stream_rows = stream_rows_from_workbook(workbook_rows)

streams_df = pd.DataFrame(
    stream_rows,
    columns=[
        'Stream Information',
        'Supply Temperature (C)',
        'Target Temperature (C)',
        'Heat Load (kW)',
        'U (kW/m2-K)',
        'CP (kW/K)',
    ],
)
streams_df.index = streams_df.index + 1
streams_df


## 2. Run Supertargeting

`run_supertargeting()` returns the key pinch-analysis and area-targeting
outputs for the selected `Delta Tmin`.


In [ ]:
streams = read_streams_from_xlsx(INPUT_XLSX)
result = run_supertargeting(streams, delta_tmin_c=DELTA_TMIN_C)
replay_result = replay_notebook_area_algorithm(streams, delta_tmin_c=DELTA_TMIN_C)

summary_df = pd.DataFrame(
    [
        {'Metric': 'Delta Tmin (C)', 'Value': DELTA_TMIN_C},
        {'Metric': 'Stream count', 'Value': len(streams)},
        {'Metric': 'Pinch hot temperature (C)', 'Value': result.pinch_hot_c},
        {'Metric': 'Pinch cold temperature (C)', 'Value': result.pinch_cold_c},
        {'Metric': 'Minimum hot utility (kW)', 'Value': result.minimum_hot_utility_kw},
        {'Metric': 'Minimum cold utility (kW)', 'Value': result.minimum_cold_utility_kw},
        {'Metric': 'Supertargeting minimum area (m2)', 'Value': result.minimum_area_m2},
        {'Metric': 'Notebook-style replay area (m2)', 'Value': replay_result.total_area_m2},
        {'Metric': 'Minimum exchangers', 'Value': result.minimum_exchangers},
    ]
)
summary_df


## 3. Compare All-Stream and Process-Only Bases

The original workflow looked at both the full stream set and a process-only
basis without the named utility rows. That comparison is useful when you want
to separate process thermal structure from utility bookkeeping.


In [ ]:
process_streams = [stream for stream in streams if stream.name not in PROCESS_ONLY_UTILITY_NAMES]
process_result = run_supertargeting(process_streams, delta_tmin_c=DELTA_TMIN_C)

total_results_df = pd.DataFrame(
    [
        {'Basis': 'All streams', 'Metric': 'Stream count', 'Value': len(streams)},
        {'Basis': 'All streams', 'Metric': 'Pinch hot temperature (C)', 'Value': result.pinch_hot_c},
        {'Basis': 'All streams', 'Metric': 'Pinch cold temperature (C)', 'Value': result.pinch_cold_c},
        {'Basis': 'All streams', 'Metric': 'Minimum hot utility (kW)', 'Value': result.minimum_hot_utility_kw},
        {'Basis': 'All streams', 'Metric': 'Minimum cold utility (kW)', 'Value': result.minimum_cold_utility_kw},
        {'Basis': 'All streams', 'Metric': 'Minimum area (m2)', 'Value': result.minimum_area_m2},
        {'Basis': 'Process only', 'Metric': 'Stream count', 'Value': len(process_streams)},
        {'Basis': 'Process only', 'Metric': 'Pinch hot temperature (C)', 'Value': process_result.pinch_hot_c},
        {'Basis': 'Process only', 'Metric': 'Pinch cold temperature (C)', 'Value': process_result.pinch_cold_c},
        {'Basis': 'Process only', 'Metric': 'Minimum hot utility (kW)', 'Value': process_result.minimum_hot_utility_kw},
        {'Basis': 'Process only', 'Metric': 'Minimum cold utility (kW)', 'Value': process_result.minimum_cold_utility_kw},
        {'Basis': 'Process only', 'Metric': 'Minimum area (m2)', 'Value': process_result.minimum_area_m2},
    ]
)
total_results_df


## 4. Inspect the Area Intervals

These are the interval-level results returned by the targeting algorithm.
They are useful for checking how the heat duty is partitioned and where the
total area estimate comes from.


In [ ]:
intervals_df = pd.DataFrame(
    [
        {
            'Delta H Interval (kW)': f'{interval.enthalpy_start_kw:.1f}--{interval.enthalpy_end_kw:.1f}',
            'Th_out (C)': interval.hot_out_c,
            'Th_in (C)': interval.hot_in_c,
            'Tc_in (C)': interval.cold_in_c,
            'Tc_out (C)': interval.cold_out_c,
            'Hot Streams': list(interval.hot_stream_ids),
            'Cold Streams': list(interval.cold_stream_ids),
            'LMTD (C)': interval.lmtd_c,
            'Area (m2)': interval.area_m2,
        }
        for interval in result.area_intervals
    ]
)
intervals_df


## 5. Plot Composite-Curve Style Summaries

`curve_plot_records()` provides the plotting coordinates for both the
composite curve and the balanced composite curve.


In [ ]:
all_curve_df = pd.DataFrame(curve_plot_records(streams, result))
process_curve_df = pd.DataFrame(curve_plot_records(process_streams, process_result))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

for curve_name, color in [('hot_curve', '#c65d2e'), ('cold_curve', '#1f4e79')]:
    subset = all_curve_df[(all_curve_df['plot_name'] == 'bcc_curve') & (all_curve_df['curve_name'] == curve_name)]
    axes[0].plot(subset['enthalpy_kw'], subset['temperature_c'], marker='o', linewidth=2.2, label=curve_name)
axes[0].set_title('Balanced Composite Curve')
axes[0].set_xlabel('Enthalpy (kW)')
axes[0].set_ylabel('Temperature (C)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

for curve_name, color in [('hot_curve', '#c65d2e'), ('cold_curve', '#1f4e79')]:
    subset = process_curve_df[(process_curve_df['plot_name'] == 'composite_curve') & (process_curve_df['curve_name'] == curve_name)]
    axes[1].plot(subset['enthalpy_kw'], subset['temperature_c'], marker='o', linewidth=2.2, label=curve_name)
axes[1].set_title('Process-Only Composite Curve')
axes[1].set_xlabel('Enthalpy (kW)')
axes[1].set_ylabel('Temperature (C)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.show()


## 6. Reference Figures from the Original Workflow

The repository also keeps the figure assets exported from the original CES
workflow. These are useful for quick comparison against the notebook output.


In [ ]:
for figure_name in [
    'bcc_all_streams_dtmin20.png',
    'bcc_all_streams_partitioned_dtmin20.png',
    'composite_curve_process_only_dtmin20.png',
    'dtmin_economics_aligned_bcc.png',
]:
    print(figure_name)
    display(Image(filename=IMAGE_DIR / figure_name, width=700))
